# J/psi to mu mu, 2023: processor test

Runs the processor on one file per sample, then looks at the cutflows, a histogram and
the warnings. Re-run it after changing anything in `jpsi_analysis/tools/`: the numbers
should only move when you meant them to.

In [ ]:
# python
import importlib
# columnar analysis and plotting
import matplotlib.pyplot as plt
from coffea import processor
# local
from jpsi_analysis import TREE_NAME
from jpsi_analysis.tools import utilities, plotting, selection, histogram, cutflow, schema
from jpsi_analysis.tools import processor as analysis_processor
from jpsi_analysis.definitions import objects, cuts, hists, weights
# Always reload the local modules to pick up changes made during development. The
# order matters: a module is reloaded after everything it imports from this package.
# (A module added later, like tools/lepton_jets.py, goes in before `objects`.)
for module in (utilities, cutflow, histogram, schema, objects, cuts, hists, weights,
               selection, analysis_processor, plotting):
    importlib.reload(module)
AnalysisSchema = schema.AnalysisSchema
plotting.set_plot_style()
%matplotlib inline

In [ ]:
samples = ["jpsi_mumu_2023_condor"]
fileset = utilities.make_fileset(samples, max_files=1)
fileset

In [ ]:
channels = ["all", "baseline", "baseline_2muons"]
hist_collections = ["base"]

runner = processor.Runner(
    executor=processor.IterativeExecutor(),
    # executor=processor.FuturesExecutor(workers=4),
    schema=AnalysisSchema,
    chunksize=100_000,
    # maxchunks=1,
    skipbadfiles=False,   # True carries on past unreadable files; in a test one should stop the run
    metadata_cache={},    # else coffea reuses the sample metadata (is_data, year, skim_factor)
                          # a file was first run with in this session, whatever the fileset says
)
p = analysis_processor.AnalysisProcessor(channels, hist_collections)
out = runner(fileset, processor_instance=p, treename=TREE_NAME)

## Cutflows

One row per event-level cut. Object cuts slim collections and do not appear here.

In [ ]:
for channel in channels:
    print(channel)
    out[samples[0]]["cutflow"][channel].print_table()
    print()

## Warnings

Everything the processor skipped, and why. A generator-level cut on data belongs here;
anything you did not expect is worth understanding before looking at plots.

In [ ]:
for sample in samples:
    for message in sorted(out[sample]["warnings"]):
        print(f"{sample}: {message}")

## Histograms

Every histogram has a `channel` axis; pick a channel to get the plain histogram.

In [ ]:
for channel in channels:
    plotting.plot(utilities.get_hist(out, samples[0], "pv_n", channel), label=channel,
                  skip_label=True)
plotting.add_label(data=True in out[samples[0]]["metadata"]["is_data"])
plt.legend()
plt.yscale("log")

In [ ]:
out[samples[0]]["metadata"]